# qust strategy analysis examples

这个 notebook 参考 vectorbt.pro 的 analysis 思路，用 qust 完成三个能直接落地的策略分析任务：

1. 从策略信号生成日频收益，并输出 `returns_stats` 统计表；
2. 把每一笔交易拆成持仓第 N 根 bar 的收益分布，并用 violin 图展示；
3. 在 K 线上标记开仓/平仓信号，再抽取完整交易行做检查。

策略固定使用：

```python
stras = stra_assets.get_all_strategy_exprs()
stra = stras[0]
```

这本 notebook 的重点不是复刻 vectorbt 的全部界面，而是展示 qust 的表达式管线：原始 K 线表进来，信号、持仓、收益、统计和图形输出都尽量在 qust 里完成。


## How to Read This Notebook

这本 notebook 按“输入 -> qust 表达式 -> 输出/图形”的方式组织，不是零散代码片段。

建议按这个顺序读：

1. 先看输入表需要哪些列，以及这些列在策略/指标里代表什么；
2. 再看 qust 表达式每一步新增、过滤、聚合了什么；
3. 最后看 `DataFrame` 输出或 qust monitor 的真实交互输出；
4. 如果要换成自己的数据，先保证列名、类型、时间粒度一致，再改参数。

所有图形单元格都保留 qust/monitor 的原始 notebook 输出，不用 PNG 截图。运行 notebook 时，图形 cell 返回什么，保存的输出就是什么。

当前主题：`strategy analysis`。


In [ ]:
import sys
sys.path.insert(0, "/root/otters/otters-py/python")

import importlib
import qust as qs
qs = importlib.reload(qs)
import qust.future.future as _qust_future_future
import qust.future as _qust_future
importlib.reload(_qust_future_future)
importlib.reload(_qust_future)
import qust_bg as bg
bg = importlib.reload(bg)

from qust import col, stra_assets
from qust import datasource as qds
from qust._polars import pl
from qust.monitor import mark_shape
from IPython.display import display

pl.Config.set_tbl_rows(14)
pl.Config.set_tbl_cols(16)

DATA_PATH = "/root/qust-py/examples/data/data_kline2.parquet"
ROWS = 120_000
PLOT_TICKER = "au"

source = qds.read_parquet(DATA_PATH, chunk_size=200_000)
data = col.all.calc_data(source).head(ROWS)
stras = stra_assets.get_all_strategy_exprs()
stra = stras[0]

summary = col(
    col("datetime").min().alias("start"),
    col("datetime").max().alias("end"),
    col("close").count().alias("rows"),
).calc_data(data)
ticker_count = col("close").count().group_by("ticker").calc_data(data).height
summary = summary.with_columns(pl.lit(ticker_count).alias("tickers"))
summary


## 1. Returns stats：从 K 线到日频收益统计

目标是得到一张类似 vectorbt returns stats 的统计表。最小输入是：

| 输入列 | 用途 |
| --- | --- |
| `ticker` | 多品种分组计算，保证每个品种独立维护状态 |
| `datetime` | 之后转成 `date` 做日频聚合 |
| `open/high/low/close/volume` | 策略、仓位和收益计算需要的行情列 |

表达式流程是：

```text
原始 K 线
-> with_cols(stra) 生成 open/exit 信号
-> stra.to_hold_two_sides().expanding() 生成逐行持仓 hold
-> hold / bg.vol_pms() 做波动归一化仓位
-> bt.price() 根据 close 和持仓计算逐行 pnl
-> over("ticker") 保证每个品种状态独立
-> group_by(date) 聚合成日频 pnl 和 benchmark
-> bt.returns_stats() 输出统计表
```

`pnl` 是策略每期收益率，小数表示；`0.01` 表示 1%。如果 `bt.returns_stats()` 输入第三列 `benchmark`，会额外计算 Benchmark Return、Alpha、Beta；只有 `date, pnl` 时，基准相关指标会是 null。


In [ ]:
strategy_daily = (
    col
    .with_cols(stra)
    .with_cols(
        col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra.to_hold_two_sides()
            .expanding()
            .alias("hold")
    )
    .with_cols(
        (col("hold") / col.all.bg.vol_pms()).alias("hold_sized"),
        col("close").pct().expanding().alias("benchmark_ret"),
    )
    .with_cols(
        col("close", "hold_sized").bt.price().expanding()
    )
    .over("ticker")
    .select(
        col(
            col("pnl").sum().alias("pnl"),
            col("benchmark_ret").mean().alias("benchmark"),
        ).group_by(col("datetime").dt.date().alias("date"))
    )
    .calc_data(data)
    .sort("date")
)

strategy_daily.head(10)


上面的 `strategy_daily` 已经从逐 bar 数据压成日频数据。下一步只需要把 `date, pnl, benchmark` 三列交给 `bt.returns_stats()`。

输出表里每一行是一个指标：

- `Total Return [%]`：策略总收益；
- `Annualized Return [%]`：按 `periods_per_year` 年化后的收益；
- `Sharpe Ratio`、`Sortino Ratio`：收益与风险的比值；
- `Max Drawdown [%]` 和 `Max Drawdown Duration`：最大回撤及持续时间；
- `Alpha`、`Beta`：相对 benchmark 的线性暴露。


In [ ]:
returns_stats = col("date", "pnl", "benchmark").bt.returns_stats(periods_per_year=252).calc_data(strategy_daily)
returns_stats


统计表给的是总体结论，曲线图负责展示过程。

下面用 qust 继续把日频 `pnl` 累加成 `strategy_equity`，把 benchmark 累加成 `benchmark_equity`，再计算回撤 `drawdown`。图形仍然是 qust monitor 的真实输出，不是截图。


In [ ]:
curve_data = (
    col.with_cols(
        (col("pnl").fill_null(col.lit(0.0)).sum().expanding() + col.lit(1.0)).alias("strategy_equity"),
        (col("benchmark").fill_null(col.lit(0.0)).sum().expanding() + col.lit(1.0)).alias("benchmark_equity"),
    )
    .with_cols(
        col("strategy_equity").bt.drawdown().alias("drawdown")
    )
    .calc_data(strategy_daily)
)

curve_dashboard = col(
    col("date", "strategy_equity", "benchmark_equity")
        .monitor("equity", show_axis_label=True)
        .line(),
    col("date", "drawdown")
        .monitor("drawdown", show_axis_label=True)
        .fill(color="#ff6b6b", opacity=0.35),
).monitor.make_monitor("black").monitor.add_grid([
    ["equity"],
    ["drawdown"],
]).runtime()

curve_dashboard.plot(curve_data, open_in_jupyter=True, auto_open=False, height=720)


In [ ]:
stats_table = col("metric", "value").monitor("returns_stats", show_axis_label=True).table().runtime()
stats_table.plot(returns_stats, open_in_jupyter=True, auto_open=False, height=640)


## 2. Expanding trade metrics：看持仓第 N 根 bar 的收益分布

这一段不按自然日看收益，而是按“交易生命周期”展开。它回答的问题是：一笔交易开仓之后，持仓第 1、2、3、... 根 bar 时，历史上所有交易的累计收益分布是什么样？

输入仍然是原始 K 线和策略信号。表达式流程是：

```text
原始 K 线
-> with_cols(stra) 生成信号
-> close.pct().expanding() / pms 得到归一化单 bar 收益 ret
-> stra.bin_trade() 给每一笔多头交易编号 bin_trade
-> over("ticker") 保证不同品种的交易编号独立
-> filter(bin_trade is not null) 只保留交易中的行
-> over("bin_trade") 在每笔交易内部计算 open_elapsed 和 ret_cum
-> group_by("open_elapsed").batch.violin_profile() 压缩分布 profile
-> monitor.violin() 画每个持仓时长的收益分布
```

这里先算 `batch.violin_profile()`，再画 violin。这样 monitor 拖动/缩放时不会从原始样本重新计算 KDE，分布也不会因为可见窗口变化而改变。


In [ ]:
trade_profile_expr = (
    col
       .with_cols(
           stra,
           col.all.bg.vol_pms().alias("pms")
       )
       .with_cols(
           (col("close").pct().expanding() / col("pms")).alias("ret")
       )
       .with_cols(
           col("open_long_sig", "exit_long_sig").stra.bin_trade().alias("bin_trade")
       )
       .over("ticker")
       .filter(
           col("bin_trade").is_not_null()
       )
       .with_cols(
           col(
               col("ret").count().expanding().alias("open_elapsed"),
               col("ret").sum().expanding().alias("ret_cum")
           )
               .over("bin_trade")
       )
       .filter(
           col("ret_cum").is_not_null()
       )
       .select(
           col("ret_cum")
               .batch.violin_profile()
               .group_by("open_elapsed")
               .batch.sort("open_elapsed")
       )
)

trade_profile = trade_profile_expr.calc_data(data)
trade_profile.select("open_elapsed", "count", "min", "q1", "median", "q3", "max", "is_point").head(12)


In [ ]:
trade_violin = col.all.monitor("trade_violin", show_axis_label=True).violin().runtime()
trade_violin.plot(trade_profile, open_in_jupyter=True, auto_open=False, height=680)


## 3. Trade signals：在 K 线上检查开平仓点

策略分析不能只看最终收益，必须回到信号本身检查：哪里开仓、哪里平仓、是否存在连续开仓、是否有异常空仓/持仓。

`stras[0]` 输出四列：

| 信号列 | 含义 |
| --- | --- |
| `open_long_sig` | 开多 |
| `exit_long_sig` | 平多 |
| `open_short_sig` | 开空 |
| `exit_short_sig` | 平空 |

下面先统计四类信号数量，再选一个品种 `PLOT_TICKER` 画 K 线和信号点。最后用 `stra.trade_row()` 把完整交易抽成表格，便于逐笔核对 entry/exit index、时间和价格。


In [ ]:
signal_data = col.with_cols(stra).over("ticker").calc_data(data)

signal_counts = col(
    col("open_long_sig").sum().alias("open_long"),
    col("exit_long_sig").sum().alias("exit_long"),
    col("open_short_sig").sum().alias("open_short"),
    col("exit_short_sig").sum().alias("exit_short"),
).calc_data(signal_data)

signal_counts


In [ ]:
plot_signal_data = signal_data.filter(pl.col("ticker") == PLOT_TICKER).head(8000)

signal_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("signals", show_axis_label=True)
        .kline(),
    col("datetime", "close", "open_long_sig")
        .monitor("signals")
        .mark(shape=mark_shape.triangle_up, color="#4dd0e1", width=0.35),
    col("datetime", "close", "exit_long_sig")
        .monitor("signals")
        .mark(shape=mark_shape.triangle_down, color="#f2c94c", width=0.35),
    col("datetime", "close", "open_short_sig")
        .monitor("signals")
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.35),
    col("datetime", "close", "exit_short_sig")
        .monitor("signals")
        .mark(shape=mark_shape.triangle_up, color="#b388ff", width=0.35),
).monitor.make_monitor("black").monitor.add_grid([["signals"]]).runtime()

signal_dashboard.plot(plot_signal_data, open_in_jupyter=True, auto_open=False, height=680)


In [ ]:
trade_rows = (
    col("ticker", "datetime", "close")
        .get_by_index(
            col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig").stra.trade_row(),
            keep=True,
        )
        .over("ticker")
        .calc_data(signal_data)
)

trade_rows.select(
    "entry_index", "exit_index", "side", "ticker_at_entry_index",
    "datetime_at_entry_index", "datetime_at_exit_index",
    "close_at_entry_index", "close_at_exit_index",
).head(12)
